# BC카드 추천 엔진 - 카테고리별 지출 기반 최적 카드 추천 프로토타입

## 1. 필요 라이브러리 및 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 전처리된 카드 혜택 데이터 불러오기
df_benefit = pd.read_csv('bc_card_benefit_preprocessed_v2.csv')

# 데이터 확인
print("전체 데이터 행 수:", len(df_benefit))
print("수집된 카드 수:", df_benefit['card_name'].nunique())
print("\n데이터 컬럼:")
print(df_benefit.columns.tolist())

# benefit_category_v2 컬럼이 있으면 benefit_category로 사용
if 'benefit_category' not in df_benefit.columns and 'benefit_category_v2' in df_benefit.columns:
    df_benefit['benefit_category'] = df_benefit['benefit_category_v2']
elif 'benefit_category' not in df_benefit.columns:
    # 임시로 benefit_category 생성
    df_benefit['benefit_category'] = '기타'

print("\n샘플 데이터:")
sample_cols = ['card_name', 'benefit_title', 'benefit_type', 'benefit_rate_v2']
display(df_benefit[sample_cols].head(10))

전체 데이터 행 수: 831
수집된 카드 수: 139

데이터 컬럼:
['card_name', 'affiFirmNo', 'mbNo', 'main_benefit', 'detail_url', 'benefit_order', 'benefit_title', 'sub_titles', 'list_texts', 'notes', 'full_text', 'raw_text', 'benefit_type', 'benefit_rate_v2', 'all_monthly_limits', 'all_previous_month_spend', 'monthly_limit_candidates', 'spend_limit_map']

샘플 데이터:


,card_name,benefit_title,benefit_type,benefit_rate_v2
0,[BNK경남] 경남은행 K-패스 카드,모빌리티 할인 서비스(결제일 차감 청구),할인,15.0
1,[BNK경남] 경남은행 K-패스 카드,생활 할인 서비스(결제일 차감 청구),할인,5.0
2,[BNK경남] 경남은행 K-패스 카드,기타 안내,기타,NaN
3,[Sh수협] 더 아우름 카드,프리미엄 바우처 (Premium Voucher),할인,NaN
4,[Sh수협] 더 아우름 카드,할인서비스,할인,5.0
5,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(공통),할인,15.0
6,[Sh수협] 더 아우름 카드,마스터카드 월드서비스(선택),할인,NaN
7,[Sh수협] 더 아우름 카드,기타 안내,바우처,NaN
8,[BC바로] BC 바로 ZONE 카드,"EAT ZONE 커피, 간식, 배달앱 7% 할인",할인,7.0
9,[BC바로] BC 바로 ZONE 카드,"LIFE ZONE 온라인몰, 다이소, 올리브영 7% 할인",할인,7.0


## 2. 사용자 입력: 월 지출 총액 및 카테고리별 지출 입력

In [3]:
# ========================
# 사용자 입력 설정
# ========================

# 예시: 사용자 월 지출 카테고리별 입력
# 실제 사용할 때는 여기를 수정하시면 됩니다.

user_spending = {
    '쇼핑': 200000,          # 쇼핑 카테고리 20만원
    '교통': 200000,          # 교통 카테고리 20만원
    '음식점': 300000,         # 음식점(푸드) 카테고리 30만원
    '카페': 100000,           # 카페 10만원
    '편의점': 50000,          # 편의점 5만원
}

print("=" * 60)
print("📊 사용자 월 지출 구성")
print("=" * 60)

total_spending = sum(user_spending.values())
for category, amount in sorted(user_spending.items(), key=lambda x: x[1], reverse=True):
    print(f"{category:12} : {amount:>10,}원 ({amount/total_spending*100:5.1f}%)")
print("-" * 60)
print(f"총 월 지출액   : {total_spending:>10,}원")
print("=" * 60)

📊 사용자 월 지출 구성
음식점          :    300,000원 ( 35.3%)
쇼핑           :    200,000원 ( 23.5%)
교통           :    200,000원 ( 23.5%)
카페           :    100,000원 ( 11.8%)
편의점          :     50,000원 (  5.9%)
------------------------------------------------------------
총 월 지출액   :    850,000원


## 3. 카드별 혜택 계산 함수 구현

In [6]:
import ast
import re

def parse_list_column(value):
    """문자열로 저장된 리스트 파싱"""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        return ast.literal_eval(str(value))
    except:
        return []

def extract_rate_from_text(text):
    """텍스트에서 할인/적립 비율 추출"""
    if pd.isna(text):
        return None
    
    patterns = [
        r"(\d+(?:\.\d+)?)\s*%",
        r"(\d+(?:\.\d+)?)\s*퍼센트"
    ]
    
    for p in patterns:
        m = re.search(p, str(text))
        if m:
            return float(m.group(1))
    
    return None

def calculate_benefit_for_card(card_row, user_spending, total_spending):
    """
    한 장의 카드에 대해 사용자 지출에 따른 월간 예상 혜택 금액 계산
    
    Parameters:
    - card_row: 카드의 혜택 정보 (DataFrame 행)
    - user_spending: 사용자의 카테고리별 월 지출 {'카테고리': 금액}
    - total_spending: 총 월 지출액
    
    Returns:
    - benefit_amount: 월간 예상 혜택 금액
    - details: 계산 상세 정보
    """
    
    benefit_amount = 0
    details = []
    
    # 혜택 정보 추출
    benefit_title = card_row.get('benefit_title', '')
    raw_text = card_row.get('raw_text', '')
    benefit_type = card_row.get('benefit_type', '')
    
    # 할인/적립 비율 추출 (기본: raw_text에서 추출, 없으면 전체 텍스트 사용)
    benefit_rate = card_row.get('benefit_rate_v2')
    if pd.isna(benefit_rate) or benefit_rate == 0:
        benefit_rate = extract_rate_from_text(raw_text)
    
    # 한도 정보
    monthly_limits = parse_list_column(card_row.get('monthly_limit_candidates', []))
    prev_spend = parse_list_column(card_row.get('all_previous_month_spend', []))
    
    # Rate가 없으면 혜택 없음
    if pd.isna(benefit_rate) or benefit_rate == 0:
        return 0, details
    
    # 카테고리 매칭 (benefit_title, raw_text, benefit_type에서 찾기)
    matched_categories = []
    
    search_text = f"{benefit_title} {str(raw_text)[:500]}".lower()
    
    for user_cat, user_amount in user_spending.items():
        cat_keywords = [user_cat.lower()]
        
        # 카테고리별 추가 키워드
        category_keywords = {
            '쇼핑': ['쇼핑', '온라인', '이커머스'],
            '교통': ['교통', '버스', '지하철', '택시', '모빌리티'],
            '카페': ['카페', '커피', '스타벅스', '투썸', '이디야'],
            '음식점': ['음식', '외식', '배달', '레스토랑'],
            '편의점': ['편의점', 'cu', 'gs', '세븐'],
            '푸드': ['음식', '외식', '배달', '레스토랑'],
        }
        
        if user_cat in category_keywords:
            cat_keywords.extend(category_keywords[user_cat])
        
        for keyword in cat_keywords:
            if keyword in search_text:
                matched_categories.append((user_cat, user_amount))
                break
    
    # 중복 제거 (같은 카테고리)
    matched_categories = list(set(matched_categories))
    
    # 매칭된 카테고리가 없으면 혜택 없음
    if not matched_categories:
        return 0, details
    
    # 카테고리별 혜택 계산
    for user_cat, user_amount in matched_categories:
        
        # 전월실적 조건 체크
        spend_condition_met = True
        required_spend = 0
        
        if len(prev_spend) > 0:
            # 전월실적이 있는 경우 가장 낮은 조건을 기준으로 함
            required_spend = min(prev_spend)
            spend_condition_met = (total_spending >= required_spend)
        
        # 전월실적 조건 미충족 시 혜택 없음
        if not spend_condition_met:
            continue
        
        # 한도 정보 추출
        applicable_limit = None
        
        if len(monthly_limits) > 0:
            if len(prev_spend) > 0 and len(monthly_limits) == len(prev_spend):
                # 실적과 한도가 일대일 대응
                for spend, limit in zip(sorted(prev_spend), sorted(monthly_limits)):
                    if total_spending >= spend:
                        applicable_limit = limit
            elif len(monthly_limits) == 1:
                # 한도만 있음
                applicable_limit = monthly_limits[0]
        
        # 혜택 계산
        calc_benefit = (user_amount * benefit_rate / 100)
        
        # 한도 적용
        if applicable_limit:
            calc_benefit = min(calc_benefit, applicable_limit)
        
        benefit_amount += calc_benefit
        
        details.append({
            'category': user_cat,
            'spending': user_amount,
            'rate': benefit_rate,
            'limit': applicable_limit,
            'benefit': calc_benefit,
            'spend_required': required_spend
        })
    
    return benefit_amount, details


# 함수 테스트
print("✅ 혜택 계산 함수 준비 완료")

✅ 혜택 계산 함수 준비 완료


## 4. 카드별 예상 월간 혜택 금액 계산

In [7]:
print("🔄 모든 카드에 대해 혜택을 계산 중입니다...\n")

# 총 월 지출액
total_spending = sum(user_spending.values())

# 각 카드별 예상 월간 혜택 계산
results = []

for card_name in df_benefit['card_name'].unique():
    card_benefits = df_benefit[df_benefit['card_name'] == card_name]
    
    total_card_benefit = 0
    all_details = []
    
    # 카드의 모든 혜택을 합산
    for idx, row in card_benefits.iterrows():
        benefit, details = calculate_benefit_for_card(row, user_spending, total_spending)
        total_card_benefit += benefit
        all_details.extend(details)
    
    # 혜택이 있는 카드만 결과에 추가
    if total_card_benefit > 0 or len(all_details) > 0:
        results.append({
            'card_name': card_name,
            'monthly_benefit': total_card_benefit,
            'benefit_details': all_details,
            'benefit_count': len(card_benefits)
        })

# DataFrame으로 변환
df_results = pd.DataFrame(results)

print(f"✅ 계산 완료! 혜택 있는 카드: {len(df_results)}장\n")

🔄 모든 카드에 대해 혜택을 계산 중입니다...

✅ 계산 완료! 혜택 있는 카드: 121장



## 5. 카드 추천 결과 정렬 및 상위 카드 출력

In [8]:
# 월간 혜택 금액 기준으로 내림차순 정렬
df_results_sorted = df_results.sort_values('monthly_benefit', ascending=False).reset_index(drop=True)

# 상위 15개 카드 추천
top_n = 15
df_top = df_results_sorted.head(top_n)

print("\n" + "="*80)
print("🏆 BC카드 추천 결과 (월간 예상 혜택 금액 기준)")
print("="*80)
print(f"\n총 {len(df_results)}개 카드 중 상위 {min(top_n, len(df_results_sorted))}개 추천\n")

# 랭킹 출력
for rank, (idx, row) in enumerate(df_top.iterrows(), 1):
    benefit = row['monthly_benefit']
    print(f"{rank:2d}. {row['card_name']:35} | 월 예상 혜택: {benefit:>10,.0f}원")

print("\n" + "="*80)


🏆 BC카드 추천 결과 (월간 예상 혜택 금액 기준)

총 121개 카드 중 상위 15개 추천

 1. [NH농협] 국민행복카드(신용)                   | 월 예상 혜택:    185,400원
 2. [SC제일] 국민행복카드(신용)                   | 월 예상 혜택:    183,400원
 3. [IBK기업은행] 국민행복카드(신용)                | 월 예상 혜택:    183,400원
 4. [BC바로] BC 바로 On＆Off 카드              | 월 예상 혜택:    180,000원
 5. [IBK기업은행] 일상의 기쁨(신용)                | 월 예상 혜택:    175,000원
 6. [IBK기업은행] 용인시민카드(신용)                | 월 예상 혜택:    161,500원
 7. [IBK기업은행] 참! 좋은 다이소카드(신용)           | 월 예상 혜택:    159,000원
 8. [IBK기업은행] K-패스(신용)                  | 월 예상 혜택:    132,500원
 9. [iM뱅크] iM Pet Love 카드               | 월 예상 혜택:    130,000원
10. [iM뱅크] 국민행복카드(신용)                   | 월 예상 혜택:    123,400원
11. [BNK경남] 어디로든 그린카드                   | 월 예상 혜택:    120,000원
12. [NH농협] 어디로든 그린카드                    | 월 예상 혜택:    112,800원
13. [iM뱅크] 그린카드(전국)                     | 월 예상 혜택:    112,100원
14. [IBK기업은행] I-어디로든그린카드                | 월 예상 혜택:    110,600원
15. [BC바로] 신세계 콰트로 플러스                  | 월 예상 혜택:    102,000원


## 6. 추천 카드 상세 혜택 정보

In [9]:
# 상위 5개 카드의 상세 정보 출력
top_detail_count = 5

print("\n📋 상위 {} 카드 상세 혜택 정보\n".format(min(top_detail_count, len(df_top))))

for rank, (idx, row) in enumerate(df_top.head(top_detail_count).iterrows(), 1):
    card_name = row['card_name']
    monthly_benefit = row['monthly_benefit']
    details = row['benefit_details']
    
    print("=" * 80)
    print(f"🔹 #{rank} - {card_name}")
    print("=" * 80)
    print(f"예상 월간 혜택: {monthly_benefit:,.0f}원\n")
    
    if details:
        # 상세 혜택 정보를 DataFrame으로 정렬
        detail_rows = []
        for detail in details:
            detail_rows.append({
                '카테고리': detail.get('category', '-'),
                '지출액': detail.get('spending', 0),
                '할인/적립률': f"{detail.get('rate', 0):.1f}%",
                '월한도': detail.get('limit', '무제한'),
                '예상혜택': f"{detail.get('benefit', 0):,.0f}원"
            })
        
        df_detail_info = pd.DataFrame(detail_rows)
        print("📌 카테고리별 혜택 내역:")
        print(df_detail_info.to_string(index=False))
        
        # 전월실적 조건
        spending_info = [d for d in details if d.get('spend_required', 0) > 0]
        if spending_info:
            print(f"\n💳 전월실적 조건: {spending_info[0]['spend_required']:,}원 이상 (현재 입력값 {total_spending:,}원)")
    
    print()



📋 상위 5 카드 상세 혜택 정보

🔹 #1 - [NH농협] 국민행복카드(신용)
예상 월간 혜택: 185,400원

📌 카테고리별 혜택 내역:
카테고리    지출액 할인/적립률    월한도    예상혜택
  쇼핑 200000   5.0%    NaN 10,000원
 음식점 300000  20.0%    NaN 60,000원
  카페 100000  20.0%    NaN 20,000원
  교통 200000   5.0% 1500.0  1,500원
 음식점 300000  20.0%    NaN 60,000원
  카페 100000  20.0%    NaN 20,000원
  교통 200000   5.0% 1500.0  1,500원
  교통 200000   0.2%    NaN    400원
  교통 200000   5.0%    NaN 10,000원
  쇼핑 200000   1.0%    NaN  2,000원

💳 전월실적 조건: 30,000원 이상 (현재 입력값 850,000원)

🔹 #2 - [SC제일] 국민행복카드(신용)
예상 월간 혜택: 183,400원

📌 카테고리별 혜택 내역:
카테고리    지출액 할인/적립률    월한도    예상혜택
  쇼핑 200000   5.0%    NaN 10,000원
 음식점 300000  20.0%    NaN 60,000원
  카페 100000  20.0%    NaN 20,000원
  교통 200000   5.0% 1500.0  1,500원
 음식점 300000  20.0%    NaN 60,000원
  카페 100000  20.0%    NaN 20,000원
  교통 200000   5.0% 1500.0  1,500원
  교통 200000   0.2%    NaN    400원
  교통 200000   5.0%    NaN 10,000원

💳 전월실적 조건: 30,000원 이상 (현재 입력값 850,000원)

🔹 #3 - [IBK기업은행] 국민행복카드(신용)
예상 월간 혜택: 183,400원

📌 카테고리별 혜택 내역

## 7. 전체 카드 비교표

In [10]:
# 전체 카드 비교 DataFrame
df_comparison = df_results_sorted[['card_name', 'monthly_benefit']].copy()
df_comparison.columns = ['카드명', '월간 예상 혜택(원)']
df_comparison['순위'] = range(1, len(df_comparison) + 1)
df_comparison = df_comparison[['순위', '카드명', '월간 예상 혜택(원)']]

# 월간 혜택액 포맷팅
df_comparison['월간 예상 혜택(원)'] = df_comparison['월간 예상 혜택(원)'].apply(lambda x: f"{x:,.0f}")

print("\n📊 전체 카드 월간 예상 혜택 비교 (상위 20개)\n")
display(df_comparison.head(20))


📊 전체 카드 월간 예상 혜택 비교 (상위 20개)



,순위,카드명,월간 예상 혜택(원)
0,1,[NH농협] 국민행복카드(신용),"185,400"
1,2,[SC제일] 국민행복카드(신용),"183,400"
2,3,[IBK기업은행] 국민행복카드(신용),"183,400"
3,4,[BC바로] BC 바로 On＆Off 카드,"180,000"
4,5,[IBK기업은행] 일상의 기쁨(신용),"175,000"
5,6,[IBK기업은행] 용인시민카드(신용),"161,500"
6,7,[IBK기업은행] 참! 좋은 다이소카드(신용),"159,000"
7,8,[IBK기업은행] K-패스(신용),"132,500"
8,9,[iM뱅크] iM Pet Love 카드,"130,000"
9,10,[iM뱅크] 국민행복카드(신용),"123,400"


## 8. 활용 가이드

### 사용자 지출 패턴 변경하기
위의 **"2번 섹션"**에서 `user_spending` 딕셔너리를 수정하여 본인의 실제 지출 패턴을 입력하세요.

예시:
```python
user_spending = {
    '쇼핑': 300000,      # 온라인쇼핑이 많은 경우
    '교통': 100000,      # 대중교통 이용
    '음식점': 200000,    # 외식/배달
    '카페': 150000,      # 카페 자주 이용
}
```

### 주의사항
1. **전월실적**: 카드마다 다양한 전월실적 조건(0원, 30만원, 50만원 등)이 있습니다
2. **월한도**: 혜택에는 월간 한도가 설정되어 있을 수 있습니다
3. **카테고리 매칭**: 혜택의 카테고리와 지출 카테고리가 정확히 일치해야 혜택이 적용됩니다
4. **보수적 계산**: 이 추천은 기본 조건을 가정한 보수적 계산이므로 실제와 다를 수 있습니다

## 9. 사용자 정의: 원하는 카드 더 자세히 분석하기

In [11]:
# 원하는 카드의 상세 정보 조회
# 예: 위의 상위 카드 중 정확히 알고 싶은 카드를 선택
# 아래의 card_index를 1~15 사이로 변경하여 확인하세요

card_index = 1  # 1위 카드 상세 정보

if card_index <= len(df_top):
    card_idx = card_index - 1
    selected_card = df_top.iloc[card_idx]
    card_name = selected_card['card_name']
    monthly_benefit = selected_card['monthly_benefit']
    details = selected_card['benefit_details']
    
    print("=" * 80)
    print(f"📊 {card_name} - 상세 분석")
    print("=" * 80)
    print(f"월간 예상 총 혜택: {monthly_benefit:,.0f}원\n")
    
    if details:
        detail_rows = []
        for detail in details:
            detail_rows.append({
                '카테고리': detail.get('category', '-'),
                '지출액': f"{detail.get('spending', 0):,.0f}원",
                '할인/적립률': f"{detail.get('rate', 0):.1f}%",
                '월한도': f"{detail.get('limit', '무제한'):,}" if isinstance(detail.get('limit'), (int, float)) else '무제한',
                '예상 혜택': f"{detail.get('benefit', 0):,.0f}원"
            })
        
        df_detail_info = pd.DataFrame(detail_rows)
        print("📌 카테고리별 혜택 내역:")
        print(df_detail_info.to_string(index=False))
        print()
        
        # 월 실적 조건
        spending_info = [d for d in details if d.get('spend_required', 0) > 0]
        if spending_info:
            print(f"💳 전월실적 조건: {spending_info[0]['spend_required']:,}원 이상")
            print(f"   (현재 입력값: {total_spending:,}원) ✅ 조건 충족" if total_spending >= spending_info[0]['spend_required'] else f"   (현재 입력값: {total_spending:,}원) ❌ 조건 미충족")
        else:
            print("💳 전월실적 조건: 없음 ✅")
else:
    print(f"❌ card_index는 1~{len(df_top)} 사이여야 합니다.")

📊 [NH농협] 국민행복카드(신용) - 상세 분석
월간 예상 총 혜택: 185,400원

📌 카테고리별 혜택 내역:
카테고리      지출액 할인/적립률   월한도   예상 혜택
  쇼핑 200,000원   5.0%   무제한 10,000원
 음식점 300,000원  20.0%   무제한 60,000원
  카페 100,000원  20.0%   무제한 20,000원
  교통 200,000원   5.0% 1,500  1,500원
 음식점 300,000원  20.0%   무제한 60,000원
  카페 100,000원  20.0%   무제한 20,000원
  교통 200,000원   5.0% 1,500  1,500원
  교통 200,000원   0.2%   무제한    400원
  교통 200,000원   5.0%   무제한 10,000원
  쇼핑 200,000원   1.0%   무제한  2,000원

💳 전월실적 조건: 30,000원 이상
   (현재 입력값: 850,000원) ✅ 조건 충족


## 10. 데이터 JSON 내보내기 (HTML용)

In [13]:
import json

# HTML용 데이터 준비
card_data = []

# 카드별 affiFirmNo 매핑 (첫 번째 출현 기준)
card_to_affi = {}
for card_name in df_benefit['card_name'].unique():
    affi = df_benefit[df_benefit['card_name'] == card_name]['affiFirmNo'].iloc[0]
    card_to_affi[card_name] = int(affi)

for idx, row in df_results_sorted.iterrows():
    card_name = row['card_name']
    card_info = {
        'rank': idx + 1,
        'card_name': card_name,
        'affiFirmNo': card_to_affi.get(card_name, 0),
        'monthly_benefit': round(row['monthly_benefit'], 0),
        'benefit_details': row['benefit_details']
    }
    card_data.append(card_info)

# 결과 데이터
export_data = {
    'user_spending': user_spending,
    'total_spending': total_spending,
    'cards': card_data,
    'summary': {
        'total_cards': len(df_benefit['card_name'].unique()),
        'cards_with_benefits': len(df_results_sorted),
        'top_benefit': round(df_results_sorted.iloc[0]['monthly_benefit'], 0) if len(df_results_sorted) > 0 else 0
    }
}

# JSON 파일로 저장
with open('card_recommendations.json', 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print("✅ JSON 데이터 내보내기 완료!")
print(f"📁 파일: card_recommendations.json")
print(f"📊 포함된 데이터:")
print(f"   - 사용자 지출: {user_spending}")
print(f"   - 총 월 지출: {total_spending:,}원")
print(f"   - 추천 카드: {len(df_results_sorted)}장")
print(f"   - 상위 카드 월 혜택: {export_data['summary']['top_benefit']:,}원")
print(f"   - 카드 이미지 포함: ✅")

✅ JSON 데이터 내보내기 완료!
📁 파일: card_recommendations.json
📊 포함된 데이터:
   - 사용자 지출: {'쇼핑': 200000, '교통': 200000, '음식점': 300000, '카페': 100000, '편의점': 50000}
   - 총 월 지출: 850,000원
   - 추천 카드: 121장
   - 상위 카드 월 혜택: 185,400.0원
   - 카드 이미지 포함: ✅
